In [ ]:
# 허깅페이스 데이터세트 클래스에 대해
'''
허깅페이스 데이터세트에는 DatasetDict 클래스와 Dataset 클래스로 구성되어 있다.

Dataset({
          features: ['f1', 'f2'],
          num_rows : ~
        })

DatasetDict({
              train: Dataset,
              validation: Dataset,
              test: Dataset
            })

기본적인 인덱싱이 가능하다.

파이토치 데이터세트 : DataLoader와 연동할 수 있는 최소한의 장치만 구현.
메모리를 RAM에서 관리

허깅페이스 데이터세트 : 파이토치 Dataset에 부가적인 기능 구현
토크나이징, 전처리, 메모리 관리 차원에서 유리
'''
# 판다스 데이터프레임으로부터 허깅페이스로 변환하는 방법
'''
DataFrame > Dataset : Dataset.from_pandas(df, preserve_index = False)
Dataset > DatasetDict : DatasetDict({'train':, 'validation':, 'test':})
Dataset > DataFrame : Dataset.select(range).to_pandas()
'''
# DatasetDict의 활용: 일괄 전처리
'''
datasets = dataset_dict.map(preprocess_fn)
datasets = dataset_dict.filter(filter_fn)

내부의 함수들은 dataset[0], dataset[1], ... 루프를 타며 함수를 적용한다.


'''
# 1. filter() 함수 정의 방법
'''
dataset[0],[1] 등은 {'en':, 'de':} 의 딕셔너리로 이루어져 있으므로

def filter_short_sentences(example):
    return len(example['en']) >= 5

'''
# 2. map() 함수 정의 방법
'''
전처리 함수를 적용한다.
# 토크나이저 로드 (예: MarianMT 번역 모델용)

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

def preprocess_fn(examples):
    # 1. 입력 언어(영어) 토큰화
    model_inputs = tokenizer(
        examples['en'],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # 2. 타겟 언어(독일어) 토큰화 (labels 생성)
    labels = tokenizer(
        text_target=examples['de'],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # 3. 모델에 전달할 딕셔너리에 labels 추가
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

'''
# 자료 변환 + 원하는 열 추출
'''
newdf.set_format(type = 'torch', columns = ['age','won'])
'''

# 정리
# 1. 데이터 세트를 일정 개수로 분할
'''
# 전체에서 test를 꺼내옴
split1 = dataset.train_test_split(test_size = 2000, seed = 42)
# 나머지에서 train과 test를 다시 분리
split2 = split1['test'].train_test_split(test_size = 2000, seed = 42)

dataset_dict = DatasetDict({
    'train' = split2['train']
    'validation' = split2['test']
    'test' = split1['test']
})
'''
# 2. map() 함수와 filter() 함수를 이용하여 전처리 함
'''
tokenized_datasets = dataset_dict.map(
    preprocess_fn,
    batched=True, #C++, Rust 기반으로 작성된 Fast Tokenizer라면 속도가 증폭됨. batch_size 옵션으로 조절할 수도 있음.
    remove_columns=['en', 'de'] # 원시 텍스트 열은 더 이상 필요 없으므로 제거
)
-------------------------------------------------------------------------
def is_valid_length(example):
    # 'en' 텍스트의 길이가 10자 이상 200자 이하인 경우만 True 반환
    text = example['en']
    return 10 <= len(text) <= 200

# True인 샘플만 남김
filtered_datasets = dataset_dict.filter(is_valid_length)
'''

In [ ]:
# 허깅페이스에서 토크나이저를 선정할 때 고려해야할 점
'''
1.사전학습된 모델과의 일치성
사전학습된 모델을 파인튜닝할 경우 그 모델의 것과 완전히 동일한 토크나이저를 사용해야 한다.

2. Fast Tokenizer를 지원하는지
토크나이저는 Pure python 구현체와 Rust 구현체로 나뉨. 후자가 훨씬 빠름

3. vocab_size
한국어와 영어가 잘 포함된 토크나이저인지

4. 특수 토큰이 잘 선정되어있는지
- bos_token, eos_token, pad_token, unk_token, mask_token
- 대화 LLM의 경우, <|im_start|>, <|im_end|> 토큰을 지원하는지
- tokenizer.apply_chat_template() 과 같이 템플릿 여부를 지원하는지
- pad_token이 지정되어있지 않은 경우가 많아 토크나이저 로드 후
tokenizer.pad_token = tokenizer.eos_token처럼 직접 지정해 주어야 할 수도 있음

5. 패딩 방향
Decoder-only (LLM 생성 모델): 배치를 묶어 Generation 추론을 할 때는 패딩을 왼쪽(padding_side='left')에 적용해야 올바른 생성이 가능함.
Encoder-only (분류/인코딩 모델): 일반적으로 오른쪽(padding_side='right')에 패딩을 붙임.


사전학습을 할 경우
1. vocab_size가 3만 이하는 너무 작고, 10만 이상은 너무 크다.
2. 모델 구성시 pad_token id를 찾아서 지정해주어야 한다.
'''

# 부연 설명
'''
- 대화 LLM의 경우, <|im_start|>, <|im_end|> 토큰을 지원하는지
만약 없다면?
1. 다른 모델에 다른 특수 토큰이 있는지 확인
2.1 해당 함수를 호출하면 특수 토큰으로 알아서 변환해줌
messages = [{"role": "user", "content": "안녕!"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
2.2 특수 토큰 직접 추가

tokenizer = AutoTokenizer.from_pretrained("your-model-name")
model = AutoModelForCausalLM.from_pretrained("your-model-name")

# 1. 새 특수 토큰 정의 및 토크나이저 추가
표준 메시지 구조를 문자로 변환해주는 함수
special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

print(f"{num_added_toks}개의 특수 토큰이 추가되었습니다.")


- tokenizer.apply_chat_template() 과 같이 템플릿 여부를 지원하는지

messages = [
    {"role": "system", "content": "너는 유능한 코딩 도우미야."},
    {"role": "user", "content": "안녕! 자기소개 해줘."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
>>
<|im_start|>system
너는 유능한 코딩 도우미야.<|im_end|>
<|im_start|>user
안녕! 자기소개 해줘.<|im_end|>
<|im_start|>assistant


- pad_token이 지정되어있지 않은 경우가 많아 토크나이저 로드 후
pad_token은 배치로 묶어줄 때 사용함, 즉 배치단위로 학습할 것이라면, 문장길이가 제각각이라면  padding = True는 사용해야함
tokenizer에 여러 문장을 리스트로 넘겨줄 때 padding=True 또는 padding="max_length" 옵션을 설정하면 pad_token이 자동으로 채워짐
tokenizer에 padding = True 설정도 좋지만, 용량이 많은 경우 DataCollator를 사용하여 DataLoader에서 동적패딩을 할수 도 있음


# 배치별로 가장 긴 길이에 맞춰 동적으로 pad_token을 추가해주는 콜레이터
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    tokenized_dataset,
    batch_size=32,
    collate_fn=data_collator # 여기서 알아서 padding=True 처리가 진행됨
)

tokenizer.pad_token = tokenizer.eos_token처럼 직접 지정해 주어야 할 수도 있음


- 패딩 방향
tokenizer.padding_side = "left" 를 해주어야 함(생성 decoder 모델일 때)
left padding을 해야 [<pad>, <pad>, <pad>, Hello] : 마지막 토큰에서 다음 토큰을 예측하기 때문
'''

'\n- 대화 LLM의 경우, <|im_start|>, <|im_end|> 토큰을 지원하는지\n만약 없다면?\n1. 다른 모델에 다른 특수 토큰이 있는지 확인\n2.1 해당 함수를 호출하면 특수 토큰으로 알아서 변환해줌\nmessages = [{"role": "user", "content": "안녕!"}]\nprompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n2.2 특수 토큰 직접 추가\n\ntokenizer = AutoTokenizer.from_pretrained("your-model-name")\nmodel = AutoModelForCausalLM.from_pretrained("your-model-name")\n\n# 1. 새 특수 토큰 정의 및 토크나이저 추가\n표준 메시지 구조를 문자로 변환해주는 함수\nspecial_tokens_dict = {\'additional_special_tokens\': [\'<|im_start|>\', \'<|im_end|>\']}\nnum_added_toks = tokenizer.add_special_tokens(special_tokens_dict)\n\nprint(f"{num_added_toks}개의 특수 토큰이 추가되었습니다.")\n\n\n- tokenizer.apply_chat_template() 과 같이 템플릿 여부를 지원하는지\n\nmessages = [\n    {"role": "system", "content": "너는 유능한 코딩 도우미야."},\n    {"role": "user", "content": "안녕! 자기소개 해줘."}\n]\n\nprompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\nprint(prompt)\n>>\n<|im_start|>system\n너는 유능한 코딩

In [ ]:
# vocab_size가 58000개 . 3만과 10만 사이이므로 적절.
# max_length = 512
# padding_side = 'right'
# special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'},
# 0: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
#	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
#	58100: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
'''
truncation_side='right'는 텍스트가 지정한 최대 자릿수(max_length)보다 길 때,
문장의 어느 쪽을 자를지(Truncation) 결정하는 옵션 중 오른쪽(뒷부분)을 자르도록 설정하는 값임.

토크나이저의 기본 설정값(default)이며, 패딩 옵션(padding_side)과 마찬가지로 문장을 자르는 방향을 조절합니다.
문서 분류의 경우: right
단어 생성의 경우: left


즉 현재로서 고려할 것.
truncation_side='left'
tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    tokenized_dataset,
    batch_size=32,
    collate_fn=data_collator # 여기서 알아서 padding=True 처리가 진행됨
)
'''

"\ntruncation_side='right'는 텍스트가 지정한 최대 자릿수(max_length)보다 길 때,\n문장의 어느 쪽을 자를지(Truncation) 결정하는 옵션 중 오른쪽(뒷부분)을 자르도록 설정하는 값임.\n\n토크나이저의 기본 설정값(default)이며, 패딩 옵션(padding_side)과 마찬가지로 문장을 자르는 방향을 조절합니다.\n문서 분류의 경우: right\n단어 생성의 경우: left\n\n\n즉 현재로서 고려할 것.\ntruncation_side='left'\ntokenizer.pad_token = tokenizer.eos_token\n\ndata_collator = DataCollatorWithPadding(tokenizer=tokenizer)\n\ntrain_loader = DataLoader(\n    tokenized_dataset,\n    batch_size=32,\n    collate_fn=data_collator # 여기서 알아서 padding=True 처리가 진행됨\n)\n"

In [ ]:
'''
MAX_LEN = 128 (또는 max_length = 128) 설정은 토크나이저가 입력 텍스트를 정수 ID 시퀀스(Token)로 변환할 때,
한 문장에 허용되는 최대 토큰 개수를 128개로 제한하겠다는 설정

입력 데이터 세트를 자르는 기준임.
최종적으로 생성해내는 텍스트의 양과는 상관 x
만약 추론단계에서 입력되는 텍스트가 Max_len보다 길어지면 토크나이저가 앞 또는 뒷 부분만 잘라 모델에 전달함.
따라서 max_length를 모델의 Context Length 상한까지 확장해도 좋음



생성 단계 max_new_tokens or max_length: 모델이 끝없이 문장을 이어 쓰는 무한 루프에 빠지는 것을 막기 위한 출력 결과물의 제한

이 설정은 패딩(Padding) 및 잘라내기(Truncation) 옵션과 함께 동작하며, 모델에 입력되는 행렬(Tensor)의 길이 기준선 역할을 함

Transformer 모델의 어텐션(Self-Attention) 연산 비용은 문장 길이에 비례해 제곱($O(N^2)$)으로 늘어남.
MAX_LEN을 512에서 128로 낮추면 메모리 사용량이 획기적으로 줄어들고 학습 속도가 3~4배 이상 빨라짐.

즉, 모델의 입력(Input) 텐서 크기를 결정하는 직접적인 통제 장치
'''

In [ ]:
# 데이터 콜레이터와 데이터 로더 사용
# 데이터 콜레이터
'''
1. 동적 패딩
2. 텐서 변환
ex. [ {"input_ids": [1, 2]}, {"input_ids": [3, 4, 5]} ]
>> {"input_ids": tensor([[1, 2, 0], [3, 4, 5]])}
3. 태스크별 실시간 데이터 가공 (Task-Specific Augmentation)
마스크드 언어 모델링 (MLM): BERT 학습 시 DataCollatorForLanguageModeling을 사용하면,
배치를 구성할 때  실시간으로 토큰의 15%를 무작위로 [MASK] 토큰으로 교체하고
정답 라벨을 생성합니다.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

# 1. 모델에 입력으로 들어갈 텐서 (2938 -> [MASK] 토큰 ID인 103으로 교체됨)
    "input_ids": tensor(
        [[101, 1996, 3899, 103, 2006, 1996, 13523, 102]]
    ),
    # 2. 어텐션 마스크 (실제 토큰 1, 패딩 0)
    "attention_mask": tensor([[1, 1, 1, 1, 1, 1, 1, 1]]),
    # 3. 정답 및 Loss 계산용 라벨 텐서 (선택된 위치만 원본 ID 유지, 나머지는 -100)
    "labels": tensor([[-100, -100, -100, 2938, -100, -100, -100, -100]]),

 Seq2Seq 모델링: T5, BART 등의 모델 학습 시 DataCollatorForSeq2Seq를 사용해 타겟 텍스트를 오른쪽으로 시프트(Shift)하여
    디코더 입력 값을 자동으로 맞춰줍니다.
4. 손실 계산 예외 처리 (-100 마스킹)  정답 라벨(Labels)의 패딩된 위치나 Loss 계산에서 제외해야 하는 토큰 영역을 PyTorch CrossEntropyLoss의 기본 무시 값인 -100으로 채워,
학습 중 해당 부분이 손실 계산에 반영되지 않도록 처리합니다.


디코더 모델에서는
1. 동적 패딩과 2. 라벨 텐서를 생성함.
# 모 델 입력 텐서
    "input_ids": tensor([[201, 478, 1852, 0, 0]]),
    "attention_mask": tensor([[1,1, 1, 0, 0]]),
    # 라벨 텐서 (실제 토큰 위치는 원본 ID 유지, 패딩 위치는 -100)
    "labels": tensor([[201, 478, 1852, -100, -100]]),
}
'''